In [2]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import mlflow 
import dagshub


In [3]:
import dagshub
dagshub.init(repo_owner='akashkangule18', repo_name='SWIGGI-DELIVERY-TIME-PREDICTION', mlflow=True)

Accessing as akashkangule18

Initialized MLflow to track repo "akashkangule18/SWIGGI-DELIVERY-TIME-PREDICTION"

Repository akashkangule18/SWIGGI-DELIVERY-TIME-PREDICTION initialized!

In [4]:
data = pd.read_csv("../data/raw/uncleaned_swiggy (1).xls")
data.head()

,ID,Delivery_person_ID,Delivery_person_Age,Delivery_person_Ratings,Restaurant_latitude,Restaurant_longitude,Delivery_location_latitude,Delivery_location_longitude,Order_Date,Time_Orderd,Time_Order_picked,Weatherconditions,Road_traffic_density,Vehicle_condition,Type_of_order,Type_of_vehicle,multiple_deliveries,Festival,City,Time_taken(min)
0,0x4607,INDORES13DEL02,37,4.9,22.745049,75.892471,22.765049,75.912471,19-03-2022,11:30:00,11:45:00,conditions Sunny,High,2,Snack,motorcycle,0,No,Urban,(min) 24
1,0xb379,BANGRES18DEL02,34,4.5,12.913041,77.683237,13.043041,77.813237,25-03-2022,19:45:00,19:50:00,conditions Stormy,Jam,2,Snack,scooter,1,No,Metropolitian,(min) 33
2,0x5d6d,BANGRES19DEL01,23,4.4,12.914264,77.678400,12.924264,77.688400,19-03-2022,08:30:00,08:45:00,conditions Sandstorms,Low,0,Drinks,motorcycle,1,No,Urban,(min) 26
3,0x7a6a,COIMBRES13DEL02,38,4.7,11.003669,76.976494,11.053669,77.026494,05-04-2022,18:00:00,18:10:00,conditions Sunny,Medium,0,Buffet,motorcycle,1,No,Metropolitian,(min) 21
4,0x70a2,CHENRES12DEL01,32,4.6,12.972793,80.249982,13.012793,80.289982,26-03-2022,13:30:00,13:45:00,conditions Cloudy,High,1,Snack,scooter,1,No,Metropolitian,(min) 30


In [5]:
df = data.copy()

In [6]:
# # renames the cols

df.rename(columns = {'Delivery_person_ID':'rider_id',
                     'Delivery_person_Age':'age',
                     'Delivery_person_Ratings':'rating',
                     'Restaurant_latitude':'rest_lat',
                     'Restaurant_longitude':'rest_long',
                     'Delivery_location_latitude':'destination_lat',
                     'Delivery_location_longitude':'destination_long',
                     'Order_Date':'order_date',
                     'Time_Orderd':'order_time',
                     'Time_Order_picked':'order_picked_time',
                     'Weatherconditions':'weather_cond',
                     'Road_traffic_density':'traffic',
                     'Vehicle_condition':'vehicle_cond',
                     'Type_of_order':'order_type',
                     'Type_of_vehicle':'vehicle_type',
                     'multiple_deliveries':'multiple_orders',
                     'Festival':'festival',
                     'City':'city_type',
                     'Time_taken(min)':'delivery_time'
                    }, inplace = True
         )


In [7]:
# lowering all the values  and removing al  the unwnated spaces like trailing and leading spaces in all columns
def lower(column):
    return column.lower().strip()

In [8]:
df['rider_id'] = df['rider_id'].apply(lower)
df['weather_cond'] = df['weather_cond'].apply(lower)
df['traffic'] = df['traffic'].apply(lower)
df['order_type'] = df['order_type'].apply(lower)
df['vehicle_type'] = df['vehicle_type'].apply(lower)
df['festival'] = df['festival'].apply(lower)
df['city_type'] = df['city_type'].apply(lower)

In [9]:
# fixing the nan values from the data from all the columns

# for age
df['age']= df['age'].replace('NaN ', np.nan)

# for rating
df['rating']= df['rating'].replace('NaN ', np.nan)

# for order time
df['order_time']= df['order_time'].replace('NaN ', np.nan)

# for order date
df['order_date']= df['order_date'].replace('nan', np.nan)

# for order picked time
df['order_picked_time']= df['order_picked_time'].replace('NaN ', np.nan)

# for traffic
df['traffic']= df['traffic'].replace('nan', np.nan)

# for vaehical condotion
df['vehicle_cond']= df['vehicle_cond'].replace('NaN ', np.nan)

# for order typpe
df['order_type']= df['order_type'].replace('nan', np.nan)

# for vehicle type
df['vehicle_type']= df['vehicle_type'].replace('nan', np.nan)

# for multiple orders
df['multiple_orders']= df['multiple_orders'].replace('NaN ', np.nan)

# for festivel
df['festival']= df['festival'].replace('nan', np.nan)

# for city type
df['city_type']= df['city_type'].replace('nan', np.nan)


In [10]:
# for weather column
df['weather_cond']= df['weather_cond'].str.split().str.get(1)
df['weather_cond']= df['weather_cond'].replace('nan',np.nan)

In [11]:
# for delivery time
df['delivery_time'] = df['delivery_time'].str.split().str.get(1)

In [12]:
# Changing datatypes
df['age'] = df['age'].astype('float')
df['rating'] = df['rating'].astype('float')
df['multiple_orders'] = df['multiple_orders'].astype(float)


# for datetime columns
df['order_date'] = pd.to_datetime(df['order_date'],dayfirst=True)
df['order_time'] = pd.to_datetime(df['order_time'], format='%H:%M:%S')
df['order_picked_time'] = pd.to_datetime(df['order_picked_time'], format='%H:%M:%S')

# with output column
df['delivery_time'] = df['delivery_time'].astype('int')

In [13]:
df['city_name'] = df['rider_id'].str.split('res').str.get(0)

In [14]:
df['year'] = df['order_date'].dt.year
df['month'] = df['order_date'].dt.month
df['month_name'] = df['order_date'].dt.month_name()
df['day'] = df['order_date'].dt.day_name()


In [15]:
# creating the weekend column
def  weekend(column):
    if column == 'Saturday' or column == 'Sunday':
        return 1
    else:
        return 0


df['weekend']= df['day'].apply(weekend)

In [16]:
# now extracting hours and miniuts form the order time and order picked time
# creating new column such order_picked_dur. it will describe that after how many miniuts order was picked
# and again creating time period such as morning, afternon, evening and night

df['ordered_hour'] = df['order_time'].dt.hour
df['ordered_miniut'] = df['order_time'].dt.minute
df['order_picked_time_minute'] = df['order_picked_time'].dt.minute

# new column order_picked_dura
df['order_picked_in_minute'] = df['order_picked_time_minute'] - df['ordered_miniut']


# new column time duration
def time(hour):

    if hour  < 12:
        return 'morning'
    elif hour <  17:
        return 'afternoon'
    elif hour < 21:
        return 'evening'
    else:
        return 'night'


df['time_duration'] = df['ordered_hour'].apply(time)

In [17]:
# now calculating distance between restarunts and destination form the lattiude adn longitude
# but before that removin nan values from this columns for the clean approach

new_df = df.dropna(subset =['rest_lat','rest_long','destination_lat','destination_long'])

In [18]:
# new_column such as distance

def haversine(lat1, lon1, lat2, lon2):
    # Earth radius in km
    R = 6371

    lat1 = np.radians(lat1)
    lon1 = np.radians(lon1)
    lat2 = np.radians(lat2)
    lon2 = np.radians(lon2)

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = np.sin(dlat/2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2)**2
    c = 2 * np.arcsin(np.sqrt(a))

    return R * c

new_df = df.dropna(subset=[
    'rest_lat','rest_long','destination_lat','destination_long'
]).copy()

new_df['distance_km'] = haversine(
    new_df['rest_lat'],
    new_df['rest_long'],
    new_df['destination_lat'],
    new_df['destination_long']
).round(2)

In [19]:
# # now again cleanig the dataset
# # removing unwnated cols
new_df.drop(columns=['rest_lat','rest_long','destination_lat','destination_long','order_date','order_time','order_picked_time', 'year','order_picked_time_minute'], inplace = True)

In [21]:
# and again therr are some negative values in order_pick in minute so fixing the negative values
new_df['order_picked_in_minute'] = np.where(
    new_df['order_picked_in_minute'] < 0,
    new_df['order_picked_in_minute'] + 60,
    new_df['order_picked_in_minute']
)


# again need to convert all the categorical values into lower
new_df['month_name'] = new_df['month_name'].apply(lower)
new_df['day'] = new_df['day'].apply(lower)

In [22]:
# again creating new coulumn as distnce route
def route(distance):
    if distance < 5:
        return 'short'
    elif distance > 5 and distance < 10:
        return 'medium'
    else :
        return 'long'

new_df['distance_route']  = new_df['distance_km'].apply(route)

In [23]:
new_df.head()

,ID,rider_id,age,rating,weather_cond,traffic,vehicle_cond,order_type,vehicle_type,multiple_orders,festival,city_type,delivery_time,city_name,month,month_name,day,weekend,ordered_hour,ordered_miniut,order_picked_in_minute,time_duration,distance_km,distance_route
0,0x4607,indores13del02,37.0,4.9,sunny,high,2,snack,motorcycle,0.0,no,urban,24,indo,3,march,saturday,1,11.0,30.0,15.0,morning,3.03,short
1,0xb379,bangres18del02,34.0,4.5,stormy,jam,2,snack,scooter,1.0,no,metropolitian,33,bang,3,march,friday,0,19.0,45.0,5.0,evening,20.18,long
2,0x5d6d,bangres19del01,23.0,4.4,sandstorms,low,0,drinks,motorcycle,1.0,no,urban,26,bang,3,march,saturday,1,8.0,30.0,15.0,morning,1.55,short
3,0x7a6a,coimbres13del02,38.0,4.7,sunny,medium,0,buffet,motorcycle,1.0,no,metropolitian,21,coimb,4,april,tuesday,0,18.0,0.0,10.0,evening,7.79,medium
4,0x70a2,chenres12del01,32.0,4.6,cloudy,high,1,snack,scooter,1.0,no,metropolitian,30,chen,3,march,saturday,1,13.0,30.0,15.0,afternoon,6.21,medium


In [24]:
# dropping mentioned columns
new_df.drop(columns =['time_duration','ordered_miniut','order_picked_in_minute','month_name','day','order_type','distance_route'],inplace = True)

In [25]:
# checking for null values
new_df.isnull().sum()

ID                    0
rider_id              0
age                1854
rating             1908
weather_cond        616
traffic             601
vehicle_cond          0
vehicle_type          0
multiple_orders     993
festival            228
city_type          1200
delivery_time         0
city_name             0
month                 0
weekend               0
ordered_hour       1731
distance_km           0
dtype: int64

In [26]:
# dropping the null values and running exp- 1 AS experiment without imputation
new_df.dropna(inplace = True)

In [27]:
new_df.drop(columns = ['rider_id','ID'], inplace=True)

In [28]:
    # seperating x and y

    X = new_df.drop(columns =['delivery_time'])
    y = new_df['delivery_time']

    # now seperating columns
    num_cols = X.select_dtypes(include=['int32','int64','float64']).columns
    cat_cols = X.select_dtypes(include=['object']).columns

In [29]:
num_cols

Index(['age', 'rating', 'vehicle_cond', 'multiple_orders', 'month', 'weekend',
       'ordered_hour', 'distance_km'],
      dtype='object')

In [30]:
cat_cols

Index(['weather_cond', 'traffic', 'vehicle_type', 'festival', 'city_type',
       'city_name'],
      dtype='object')

In [ ]:
mlflow.set_experiment("exp-2 with null imputation")

with mlflow.start_run(run_name="Baseline model with null value imputation"):


    # seperating x and y

        X = new_df.drop(columns =['delivery_time'])
        y = new_df['delivery_time']

        # now seperating columns
        num_cols = X.select_dtypes(include=['int32','int64','float64']).columns
        ord_cols = ['traffic']
        ohe_cols = ['weather_cond', 'vehicle_type', 'festival', 'city_type','city_name']


        # spliting data
        from sklearn.model_selection import train_test_split
        X_train,X_test,y_train,y_test = train_test_split(X,y,test_size = 0.2, random_state = 42)



        # getting important libraries
        from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, RobustScaler,PowerTransformer
        from sklearn.experimental import enable_iterative_imputer
        from sklearn.impute import SimpleImputer, KNNImputer, IterativeImputer
        from sklearn.pipeline import Pipeline
        from sklearn.compose import ColumnTransformer
        from sklearn.linear_model import LinearRegression
        from sklearn.tree import DecisionTreeRegressor
        from sklearn.ensemble import RandomForestRegressor
        import xgboost as xgb
        from xgboost import XGBRegressor
        from lightgbm import LGBMRegressor
        from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error


        ord_encoder = OrdinalEncoder(
            categories=[['missing','low','medium','jam','high']],
            handle_unknown= 'use_encoded_value',
            unknown_value= -1
        )

        # building pipelien

        num_simple_pipe = Pipeline([
            ('imputer', SimpleImputer()),
            ("transformer",PowerTransformer(method="yeo-johnson")),
            ("scaler", RobustScaler())
        ])

        num_knn_pipe = Pipeline([
            ('imputer', KNNImputer()),
            ('transformer', PowerTransformer(method = "yeo-johnson")),
            ('scaler',RobustScaler())
        ])

        num_iterative_pipe = Pipeline([
            ("imputer",IterativeImputer()),
            ('transformer', PowerTransformer(method = "yeo-johnson")),
            ('scaler',RobustScaler())
        ])


        ord_pipe = Pipeline([
            ('imputer',SimpleImputer(strategy='most_frequent')),
            ('ord_encoder',ord_encoder)
        ])

        ohe_pipe = Pipeline([
            ('imputer',SimpleImputer(strategy= "most_frequent")),
            ('ohe_encoder', OneHotEncoder(drop="first",handle_unknown="ignore",sparse_output=False))
        ])

        transformer = ColumnTransformer([
            # replaceble
            ('num',num_simple_pipe,num_cols),
            # ------------------------------
            ('ord',ord_pipe,ord_cols),
            ('ohe',ohe_pipe,ohe_cols)

        ])

        model = Pipeline([
            ('preprocess',transformer),
            ('model',XGBRegressor())
        ])


        # so right now we dont know that which imputer is going to be correct 
        # do for this we are going to use randomized searched cv to get the correct imputer and necessary parameters

        from sklearn.model_selection import RandomizedSearchCV

        params = [
            {
                "preprocess__num": [num_simple_pipe],
                "preprocess__num__imputer__strategy": ["mean", "median"],

                "preprocess__ord__imputer__strategy": ["most_frequent", "constant"],
                "preprocess__ord__imputer__fill_value": ["missing"],

                "preprocess__ohe__imputer__strategy": ["most_frequent", "constant"],
                "preprocess__ohe__imputer__fill_value": ["missing"]
            },

            {
                "preprocess__num": [num_knn_pipe],
                "preprocess__num__imputer__n_neighbors": [3, 5, 7],

                "preprocess__ord__imputer__strategy": ["most_frequent", "constant"],
                "preprocess__ord__imputer__fill_value": ["missing"],

                "preprocess__ohe__imputer__strategy": ["most_frequent", "constant"],
                "preprocess__ohe__imputer__fill_value": ["missing"]
            },

            {
                "preprocess__num": [num_iterative_pipe],
                "preprocess__num__imputer__estimator": [
                    RandomForestRegressor(random_state=42),
                    LinearRegression()
                ],

                "preprocess__ord__imputer__strategy": ["most_frequent", "constant"],
                "preprocess__ord__imputer__fill_value": ["missing"],

                "preprocess__ohe__imputer__strategy": ["most_frequent", "constant"],
                "preprocess__ohe__imputer__fill_value": ["missing"]
            }
        ]


        rs = RandomizedSearchCV(
            estimator=model,
            param_distributions=params,
            n_iter=10,
            cv=3,
            scoring="neg_mean_absolute_error",
            n_jobs=1,
            verbose=2,
            error_score="raise",
            random_state=42
        )
        rs.fit(X_train,y_train)

        regressor = rs.best_estimator_

        y_train_pred = regressor.predict(X_train)
        y_test_pred = regressor.predict(X_test)

        # model evaluation

        from sklearn.metrics import r2_score, mean_absolute_error
        train_mae = mean_absolute_error(y_train,y_train_pred)
        train_r2 = r2_score(y_train,y_train_pred)
        test_mae = mean_absolute_error(y_test,y_test_pred)
        test_r2 = r2_score(y_test,y_test_pred)

        # logging params
        mlflow.log_params(rs.best_params_)
        xgb_params = model.named_steps['model'].get_params()
        mlflow.log_params(xgb_params)

        # log artifact
        mlflow.log_artifact("Swiggy-Exp2.ipynb")

        # logging data
        train_data = new_df
        train_data = mlflow.data.from_pandas(train_data)
        train_data = mlflow.log_input(train_data,context="trainig data")

        # log metric
        mlflow.log_metric("trainig_mae", train_mae)
        mlflow.log_metric("testing_mae", test_mae)
        mlflow.log_metric("training_r2", train_r2)
        mlflow.log_metric("testing_r2", test_r2)

        # signature
        signature = mlflow.models.infer_signature(X_train, regressor.predict(X_train))

        # log model
        mlflow.sklearn.log_model(
                sk_model= regressor,
                name = "XGBregressor",
                signature= signature,
                serialization_format= mlflow.sklearn.SERIALIZATION_FORMAT_CLOUDPICKLE
        )




        print("training mean absolute error: ", train_mae)
        print("testing mean absolute error: ",test_mae)
        print("training r2_score: ",train_r2)
        print('testing r2_score', test_r2)




Fitting 3 folds for each of 10 candidates, totalling 30 fits
[CV] END preprocess__num=Pipeline(steps=[('imputer', KNNImputer()), ('transformer', PowerTransformer()),
                ('scaler', RobustScaler())]), preprocess__num__imputer__n_neighbors=3, preprocess__ohe__imputer__fill_value=missing, preprocess__ohe__imputer__strategy=most_frequent, preprocess__ord__imputer__fill_value=missing, preprocess__ord__imputer__strategy=constant; total time=   1.4s
[CV] END preprocess__num=Pipeline(steps=[('imputer', KNNImputer()), ('transformer', PowerTransformer()),
                ('scaler', RobustScaler())]), preprocess__num__imputer__n_neighbors=3, preprocess__ohe__imputer__fill_value=missing, preprocess__ohe__imputer__strategy=most_frequent, preprocess__ord__imputer__fill_value=missing, preprocess__ord__imputer__strategy=constant; total time=   1.0s
[CV] END preprocess__num=Pipeline(steps=[('imputer', KNNImputer()), ('transformer', PowerTransformer()),
                ('scaler', RobustScale

c:\Users\Akash\swiggi-delivery-time-prediction\myenv\Lib\site-packages\mlflow\types\utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
c:\Users\Akash\swiggi-delivery-time-prediction\myenv\Lib\site-packages\mlflow\types\utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing value

🏃 View run Baseline model with null value imputation at: https://dagshub.com/akashkangule18/SWIGGI-DELIVERY-TIME-PREDICTION.mlflow/#/experiments/1/runs/0605fbc694d74b94a59789848efa823c
🧪 View experiment at: https://dagshub.com/akashkangule18/SWIGGI-DELIVERY-TIME-PREDICTION.mlflow/#/experiments/1


MlflowException: The sklearn model could not be serialized in the skops serialization format. skops does not support custom functions or classes that are not defined at the top level. To work around this limitation, you can set the serialization_format 'cloudpickle', while exercising caution due to the possible arbitrary code during model deserialization using CloudPickle.

training mean absolute error:  2.6598575115203857
testing mean absolute error:  3.14931583404541
training r2_score:  0.8730080723762512
testing r2_score 0.8268244862556458
